# Architecture Design for Maze Solver

## 1. Analysis of Lab3IA
- **Reusable:** `TreeNode` structure (adapted for coordinates), Queue structures (`FIFOQueue`, `LIFOQueue`, `PriorityQueue` which can be adapted slightly).
- **Adapted:** Graph representation changes to a Grid representation (matrix). We need coordinate-based neighbor generation instead of a predefined edge list (`grafo`). The general search loop structure can be adapted.
- **Implemented from scratch:** Maze loading from `.txt`, Start/Goal location in a matrix, Grid-based valid neighbor generation respecting priority (Up, Right, Down, Left), Heuristic functions strictly for 2D grids (Manhattan, Euclidean), random valid start position generation, metrics tracking (branching factor calculation, timing).

## 2. Project Architecture

The notebook will be structured logically into the following sections:

1. **Imports & Initial Setup**: Required standard libraries (`time`, `math`, `random`).
2. **Maze Loading & Utilities**: Functions to read the `.txt` file, convert to a 2D array, and find Start (`2`) and Goal (`3`).
3. **Core Data Structures**: 
   - Node (`TreeNode`) adapted with `(x, y)` state.
   - Priority and standard queues tailored for the algorithms.
4. **Grid Mechanics**: 
   - Define movement directions (Up, Right, Down, Left).
   - Generate valid neighbors (check bounds and walls `1`).
5. **Heuristics**:
   - Manhattan and Euclidean distance functions.
6. **Search Algorithms Base**:
   - A modular search engine that accepts different queuing strategies and heuristics to execute BFS, DFS, Greedy, and A*.
7. **Metrics Collection**:
   - Wrapper or integrated counters for execution time, max memory (or nodes expanded), path length, and branching factor.
8. **Experiments**:
   - Scenario 1: Predefined Start to Goal.
   - Scenario 2: Random Start to Goal.
9. **Results & Comparison**: tabulated outputs.

---
Let's begin with **Step 1: Load the maze** into a matrix natively.

In [1]:
# STEP 1: Load the maze

# load the text file and convert to integer matrix
def load_maze(file_path):
    maze = []
    # read file lines and parse integers
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            row = [int(char) for char in line.strip() if char.isdigit()]
            if row:
                maze.append(row)
    return maze

# verify loading (assuming test_maze.txt exists)
# maze_grid = load_maze('test_maze.txt')
# print(f"Maze dimensions: {len(maze_grid)}x{len(maze_grid[0]) if maze_grid else 0}")


In [7]:
# STEP 2, 3 & 4: Mechanics and grid interpretation

# find specific value in grid
def find_position(maze, value):
    for row_idx, row in enumerate(maze):
        for col_idx, cell_value in enumerate(row):
            if cell_value == value:
                return (row_idx, col_idx)
    return None

# define valid moves with UP, RIGHT, DOWN, LEFT priority
# defined as (row_offset, col_offset)
DIRECTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]

# generate valid neighbors
def get_neighbors(maze, position):
    row_idx, col_idx = position
    neighbors = []
    
    total_rows = len(maze)
    total_cols = len(maze[0]) if total_rows > 0 else 0
    
    for row_offset, col_offset in DIRECTIONS:
        neighbor_row = row_idx + row_offset
        neighbor_col = col_idx + col_offset
        
        # check if neighbor coordinates are within grid limits
        is_within_bounds = 0 <= neighbor_row < total_rows and 0 <= neighbor_col < total_cols
        
        # check bounds and verify it is not a wall
        if is_within_bounds and maze[neighbor_row][neighbor_col] != 1:
            neighbors.append((neighbor_row, neighbor_col))
            
    return neighbors

In [3]:
# STEP 5: Reusing and adapting the tree node for pathfinding

class TreeNode:
    def __init__(self, position, path_cost=0, heuristic=0, parent=None):
        self.position = position  # (row, col) tuple
        self.path_cost = path_cost  # g(n)
        self.heuristic = heuristic  # h(n)
        self.parent = parent  # allows path reconstruction
        
    def total_cost(self):
        # f(n) = g(n) + h(n)
        return self.path_cost + self.heuristic

    def __lt__(self, other):
        # tie breaker for priority queues (not strictly defined, but useful)
        return self.total_cost() < other.total_cost()

    def __repr__(self):
        return f"Node({self.position}, g: {self.path_cost}, h: {self.heuristic})"

# extract path to goal
def reconstruct_path(node):
    path = []
    current = node
    while current:
        path.append(current.position)
        current = current.parent
    return path[::-1]  # reverse to start->goal

---

**Check-in!**
We have successfully implemented Steps 1 through 5. The project architecture relies on reusing standard Queue/Node approaches while adapting matrix mechanics. 
Please review and confirm to proceed with Step 6 (BFS)!

In [4]:
# STEP 6: Implement BFS

import time

# standard queue for BFS
class FIFOQueue:
    def __init__(self):
        self.items = []
        
    def is_empty(self):
        return len(self.items) == 0
        
    def push(self, item):
        self.items.append(item)
        
    def pop(self):
        return self.items.pop(0) if not self.is_empty() else None

# basic BFS algorithm
def bfs(maze, start_pos, goal_pos):
    # setup starting node
    start_node = TreeNode(start_pos)
    frontier = FIFOQueue()
    frontier.push(start_node)
    
    # initialize metrics
    explored_nodes = 0
    visited = set()
    start_time = time.time()
    
    while not frontier.is_empty():
        current_node = frontier.pop()
        explored_nodes += 1
        
        # goal check
        if current_node.position == goal_pos:
            exec_time = time.time() - start_time
            path = reconstruct_path(current_node)
            return path, explored_nodes, exec_time
            
        visited.add(current_node.position)
        
        # expand neighbors
        for neighbor_pos in get_neighbors(maze, current_node.position):
            if neighbor_pos not in visited:
                # avoid duplicate pushes
                visited.add(neighbor_pos)
                child = TreeNode(neighbor_pos, path_cost=current_node.path_cost + 1, parent=current_node)
                frontier.push(child)
                
    exec_time = time.time() - start_time
    return None, explored_nodes, exec_time


In [5]:
# STEP 7: Implement DFS

# standard LIFO stack for DFS
class LIFOQueue:
    def __init__(self):
        self.items = []
        
    def is_empty(self):
        return len(self.items) == 0
        
    def push(self, item):
        self.items.append(item)
        
    def pop(self):
        return self.items.pop() if not self.is_empty() else None

# basic DFS algorithm
def dfs(maze, start_pos, goal_pos):
    # setup starting node
    start_node = TreeNode(start_pos)
    frontier = LIFOQueue()
    frontier.push(start_node)
    
    # initialize metrics
    explored_nodes = 0
    visited = set()
    start_time = time.time()
    
    while not frontier.is_empty():
        current_node = frontier.pop()
        explored_nodes += 1
        
        # goal check
        if current_node.position == goal_pos:
            exec_time = time.time() - start_time
            path = reconstruct_path(current_node)
            return path, explored_nodes, exec_time
            
        visited.add(current_node.position)
        
        # expand neighbors
        # reversing neighbors to maintain Up, Right, Down, Left priority when popping
        neighbors = get_neighbors(maze, current_node.position)
        for neighbor_pos in reversed(neighbors):
            if neighbor_pos not in visited:
                child = TreeNode(neighbor_pos, path_cost=current_node.path_cost + 1, parent=current_node)
                frontier.push(child)
                
    exec_time = time.time() - start_time
    return None, explored_nodes, exec_time


In [6]:
# STEP 8: Heuristics

import math

# calculate manhattan distance (|x1 - x2| + |y1 - y2|)
def manhattan_distance(pos1, pos2):
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

# calculate straight-line euclidean distance
def euclidean_distance(pos1, pos2):
    return math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)


In [8]:
# STEP 9: Greedy Best First Search
import heapq

# Priority Queue implementation wrapped for Clean Code
class PriorityQueue:
    def __init__(self):
        self.elements = []
        
    def is_empty(self):
        return len(self.elements) == 0
        
    def push(self, item):
        # heapq sorts based on the first tuple value by default
        heapq.heappush(self.elements, item)
        
    def pop(self):
        return heapq.heappop(self.elements) if not self.is_empty() else None

# basic Greedy Search algorithm
def greedy_search(maze, start_pos, goal_pos, heuristic_func):
    start_heuristic = heuristic_func(start_pos, goal_pos)
    start_node = TreeNode(start_pos, heuristic=start_heuristic)
    
    frontier = PriorityQueue()
    # for Greedy we only care about heuristic value
    frontier.push((start_node.heuristic, start_node))
    
    explored_nodes = 0
    visited = set()
    start_time = time.time()
    
    while not frontier.is_empty():
        # pop the node with best heuristic
        current_priority, current_node = frontier.pop()
        explored_nodes += 1
        
        if current_node.position == goal_pos:
            exec_time = time.time() - start_time
            path = reconstruct_path(current_node)
            return path, explored_nodes, exec_time
            
        visited.add(current_node.position)
        
        for neighbor_pos in get_neighbors(maze, current_node.position):
            if neighbor_pos not in visited:
                # avoid duplicate pushes
                visited.add(neighbor_pos)
                h_val = heuristic_func(neighbor_pos, goal_pos)
                child = TreeNode(neighbor_pos, parent=current_node, heuristic=h_val)
                # Greedy sorts strictly by h_val
                frontier.push((h_val, child))
                
    exec_time = time.time() - start_time
    return None, explored_nodes, exec_time

In [9]:
# STEP 10: A* Search Algorithm

def a_star_search(maze, start_pos, goal_pos, heuristic_func):
    start_heuristic = heuristic_func(start_pos, goal_pos)
    start_node = TreeNode(start_pos, path_cost=0, heuristic=start_heuristic)
    
    frontier = PriorityQueue()
    # A* uses total_cost = g(n) + h(n) as its priority
    frontier.push((start_node.total_cost(), start_node))
    
    explored_nodes = 0
    visited = {} # maps position to best g(n) found so far to prevent expensive re-expansions
    visited[start_pos] = 0
    
    start_time = time.time()
    
    while not frontier.is_empty():
        current_priority, current_node = frontier.pop()
        explored_nodes += 1
        
        # goal check
        if current_node.position == goal_pos:
            exec_time = time.time() - start_time
            path = reconstruct_path(current_node)
            return path, explored_nodes, exec_time
            
        # explore neighbors
        for neighbor_pos in get_neighbors(maze, current_node.position):
            # new g(n) is current g(n) + 1 (cost of moving 1 grid cell)
            new_cost = current_node.path_cost + 1
            
            # if neighbor not visited OR we found a cheaper path to it
            if neighbor_pos not in visited or new_cost < visited[neighbor_pos]:
                visited[neighbor_pos] = new_cost
                
                h_val = heuristic_func(neighbor_pos, goal_pos)
                child = TreeNode(neighbor_pos, path_cost=new_cost, heuristic=h_val, parent=current_node)
                
                # Push based on f(n) = g(n) + h(n)
                frontier.push((child.total_cost(), child))
                
    exec_time = time.time() - start_time
    return None, explored_nodes, exec_time

In [14]:
# STEP 11: Metrics collection wrapper

# wraps execution and computes additional metrics like Branching Factor
def run_and_measure(algorithm_name, maze, start_pos, goal_pos, search_func, *args):
    # Execute the algorithm
    path, explored, exec_time = search_func(maze, start_pos, goal_pos, *args)
    
    path_length = len(path) if path else 0
    
    # approximate branching factor: N = (b^(d+1) - 1) / (b - 1)
    # simplified approximation: root of explored nodes based on depth (path length)
    branching_factor = 0
    if path_length > 1:
        branching_factor = explored ** (1 / path_length)
        
    return {
        "Algorithm": algorithm_name,
        "Path Length": path_length,
        "Explored Nodes": explored,
        "Time (s)": exec_time,
        "Branching Factor (avg)": round(branching_factor, 4)
    }

# function to print a table of results cleanly
def print_comparison_table(results):
    print(f"{'Algorithm':<25} | {'Path Len':<10} | {'Explored':<10} | {'Time(s)':<10} | {'Branching Factor'}")
    print("-" * 80)
    for res in results:
        print(f"{res['Algorithm']:<25} | {res['Path Length']:<10} | {res['Explored Nodes']:<10} | "
              f"{res['Time (s)']:<10.5f} | {res['Branching Factor (avg)']}")

In [12]:
# STEP 12: Run simulations

import random

# Scenario 1: Fixed starts from maze definition
def run_scenario_1(maze_file):
    print("--- SCENARIO 1: Fixed Start to Goal ---")
    maze = load_maze(maze_file)
    if not maze:
        print("Failed to load maze.")
        return
        
    start_pos = find_position(maze, 2)
    goal_pos = find_position(maze, 3)
    
    if not start_pos or not goal_pos:
        print("Could not find start (2) or goal (3) in the maze.")
        return
        
    results = []
    
    # BFS
    results.append(run_and_measure("BFS", maze, start_pos, goal_pos, bfs))
    
    # DFS
    results.append(run_and_measure("DFS", maze, start_pos, goal_pos, dfs))
    
    # Greedy (Manhattan)
    results.append(run_and_measure("Greedy (Manhattan)", maze, start_pos, goal_pos, greedy_search, manhattan_distance))
    
    # Greedy (Euclidean)
    results.append(run_and_measure("Greedy (Euclidean)", maze, start_pos, goal_pos, greedy_search, euclidean_distance))
    
    # A* (Manhattan)
    results.append(run_and_measure("A* (Manhattan)", maze, start_pos, goal_pos, a_star_search, manhattan_distance))
    
    # A* (Euclidean)
    results.append(run_and_measure("A* (Euclidean)", maze, start_pos, goal_pos, a_star_search, euclidean_distance))
    
    print_comparison_table(results)

# utility to generate a random valid position (free path = 0)
def generate_random_position(maze):
    total_rows = len(maze)
    total_cols = len(maze[0])
    while True:
        r = random.randint(0, total_rows - 1)
        c = random.randint(0, total_cols - 1)
        if maze[r][c] == 0:
            return (r, c)

# Scenario 2: Random valid start
def run_scenario_2(maze_file, num_iterations=1):
    print(f"\n--- SCENARIO 2: Random Starts ({num_iterations} runs) ---")
    maze = load_maze(maze_file)
    goal_pos = find_position(maze, 3)
    
    if not goal_pos:
        print("Could not find goal (3) in the maze.")
        return
        
    for i in range(num_iterations):
        random_start = generate_random_position(maze)
        print(f"\nRun {i+1}: Start Pos = {random_start}")
        
        results = []
        results.append(run_and_measure("BFS", maze, random_start, goal_pos, bfs))
        results.append(run_and_measure("DFS", maze, random_start, goal_pos, dfs))
        results.append(run_and_measure("Greedy (Manhattan)", maze, random_start, goal_pos, greedy_search, manhattan_distance))
        results.append(run_and_measure("Greedy (Euclidean)", maze, random_start, goal_pos, greedy_search, euclidean_distance))
        results.append(run_and_measure("A* (Manhattan)", maze, random_start, goal_pos, a_star_search, manhattan_distance))
        results.append(run_and_measure("A* (Euclidean)", maze, random_start, goal_pos, a_star_search, euclidean_distance))
        
        print_comparison_table(results)


In [15]:
# STEP 13: Generate comparison table (Execution of experiments)
# Assume 'test_maze.txt' exists in the very same folder

file_name = 'test_maze.txt'

# Execute fixed start scenario
run_scenario_1(file_name)

# Execute random start scenario (1 time for brevity, can be changed easily)
run_scenario_2(file_name, num_iterations=1)

--- SCENARIO 1: Fixed Start to Goal ---
Algorithm                 | Path Len   | Explored   | Time(s)    | Branching Factor
--------------------------------------------------------------------------------
BFS                       | 129        | 665        | 0.00132    | 1.0517
DFS                       | 129        | 475        | 0.00078    | 1.0489
Greedy (Manhattan)        | 129        | 239        | 0.00049    | 1.0434
Greedy (Euclidean)        | 133        | 402        | 0.00094    | 1.0461
A* (Manhattan)            | 129        | 565        | 0.00153    | 1.0503
A* (Euclidean)            | 129        | 600        | 0.00172    | 1.0508

--- SCENARIO 2: Random Starts (1 runs) ---

Run 1: Start Pos = (3, 45)
Algorithm                 | Path Len   | Explored   | Time(s)    | Branching Factor
--------------------------------------------------------------------------------
BFS                       | 0          | 378        | 0.00061    | 0
DFS                       | 0          | 467 